# Lecture 5: More on the SVD
## Trefethen & Bau — Numerical Linear Algebra

Python demonstrations using NumPy and Matplotlib.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Low-Rank Approximation (Eckart–Young)

The best rank-$\nu$ approximation to $A$ is obtained by truncating the SVD. The error is $\sigma_{\nu+1}$.

In [ ]:
# A matrix with prescribed singular value decay
np.random.seed(0)
n = 4
U_rand, _ = np.linalg.qr(np.random.randn(n, n))
V_rand, _ = np.linalg.qr(np.random.randn(n, n))
sigmas = np.array([10.0, 5.0, 0.1, 0.01])
A = U_rand @ np.diag(sigmas) @ V_rand.T

U, S, Vt = np.linalg.svd(A)
print("Singular values:", np.round(S, 4))

for k in range(1, n):
    A_k = (U[:, :k] * S[:k]) @ Vt[:k, :]
    err = np.linalg.norm(A - A_k, 2)
    print(f"  Rank-{k} error: {err:.6f}  (sigma_{k+1} = {S[k]:.6f})")

## SVD for Dimension Reduction: Synthetic Voting Data

We simulate congressional voting data: two parties with distinct voting patterns, plus noise. The SVD reveals the partisan structure with no a priori model.

In [ ]:
np.random.seed(42)

n_voters = 435
n_bills = 200
n_party1 = 257  # Democrats in 111th Congress
n_party2 = n_voters - n_party1  # Republicans

# Simulate votes: +1 (yea), -1 (nay), 0 (no vote)
# Party 1 tends to vote +1 on most bills, Party 2 tends to vote -1
party_label = np.concatenate([np.ones(n_party1), 2 * np.ones(n_party2)])

A = np.zeros((n_voters, n_bills))
for i in range(n_voters):
    if party_label[i] == 1:
        A[i] = np.sign(np.random.randn(n_bills) + 0.8)  # lean yea
    else:
        A[i] = np.sign(np.random.randn(n_bills) - 0.8)  # lean nay
    # Some random no-votes
    no_vote = np.random.rand(n_bills) < 0.05
    A[i, no_vote] = 0

plt.figure(figsize=(10, 5))
plt.imshow(A, aspect='auto', cmap='RdYlBu')
plt.colorbar(label='Vote')
plt.xlabel('Bill')
plt.ylabel('Voter')
plt.title('Voting matrix (yea=+1, nay=-1, no vote=0)')
plt.tight_layout()
plt.show()

In [ ]:
# Subtract each voter's mean vote
mu = A.mean(axis=1, keepdims=True)
A_centered = A - mu

# SVD of transposed (tall) matrix
B = A_centered.T
U, S, Vt = np.linalg.svd(B, full_matrices=False)

plt.figure(figsize=(7, 3.5))
plt.plot(S[:30], 'o', markersize=4)
plt.xlabel('Index')
plt.ylabel(r'$\sigma_k$')
plt.title('Singular values (first 30)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Project each voter into the first two singular directions
x1 = U[:, 0] @ B  # partisanship
x2 = U[:, 1] @ B  # bipartisanship

fig, axes = plt.subplots(2, 1, figsize=(7, 5))
axes[0].hist(x1, bins=32, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('First coefficient (partisanship)')
axes[0].set_ylabel('Frequency')

axes[1].hist(x2, bins=32, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Second coefficient (bipartisanship)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot colored by party
colors = np.array(['blue' if p == 1 else 'red' for p in party_label])

plt.figure(figsize=(7, 5))
plt.scatter(x1[party_label == 1], x2[party_label == 1],
            c='blue', s=15, alpha=0.6, label='Party 1 (Dem)')
plt.scatter(x1[party_label == 2], x2[party_label == 2],
            c='red', s=15, alpha=0.6, label='Party 2 (Rep)')
plt.xlabel('Partisanship')
plt.ylabel('Bipartisanship')
plt.title('Synthetic Congressional Voting (SVD projection)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## The Pseudoinverse $A^+$

$A^+ = V \Sigma^+ U^T$ where $\Sigma^+$ inverts nonzero singular values and transposes.

In [ ]:
# Tall matrix example
A = np.array([[1, 0],
              [0, 2],
              [0, 0]], dtype=float)

A_pinv = np.linalg.pinv(A)
print("A =")
print(A)
print("\nA+ =")
print(A_pinv)
print("\nA+ @ A (should be I_2):")
print(np.round(A_pinv @ A, 10))

In [ ]:
# Rank-deficient square matrix
A = np.array([[1, 2],
              [2, 4]], dtype=float)

U, S, Vt = np.linalg.svd(A)
print(f"Singular values: {S}")
print(f"Rank: {np.sum(S > 1e-10)}")

A_pinv = np.linalg.pinv(A)
print(f"\nA+ =\n{A_pinv}")
print(f"\nA @ A+ @ A (should equal A):\n{np.round(A @ A_pinv @ A, 10)}")

## Pseudoinverse Solves Least Squares

### Overdetermined system

$A = \begin{pmatrix}1\\1\\1\end{pmatrix}$, $b = \begin{pmatrix}2\\3\\4\end{pmatrix}$ — the pseudoinverse gives the mean.

In [ ]:
A = np.array([[1], [1], [1]], dtype=float)
b = np.array([2, 3, 4], dtype=float)

x_plus = np.linalg.pinv(A) @ b
print(f"x+ = {x_plus}  (= mean of b = {b.mean()})")
print(f"Ax+ = {(A @ x_plus).ravel()}")
print(f"Residual ||Ax+ - b|| = {np.linalg.norm(A @ x_plus - b):.4f}")

In [ ]:
# Visualize: projection of b onto column space
fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection='3d')

origin = [0, 0, 0]
ax.quiver(*origin, *b, color='blue', arrow_length_ratio=0.05, linewidth=2, label='b = (2,3,4)')
Ax = (A @ x_plus).ravel()
ax.quiver(*origin, *Ax, color='green', arrow_length_ratio=0.07, linewidth=2, label=r'$Ax^+$ = (3,3,3)')
ax.quiver(*Ax, *(b - Ax), color='red', arrow_length_ratio=0.1, linewidth=2, label='residual')

# Column space line
t_line = np.linspace(0, 5, 50)
ax.plot(t_line, t_line, t_line, 'g--', alpha=0.3, label='column space')

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('z')
ax.set_title('Least squares: projection onto column space')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

### Underdetermined system

$A = \begin{pmatrix}1&1&1\end{pmatrix}$, $b = 6$ — the pseudoinverse picks the minimum-norm solution.

In [ ]:
A = np.array([[1, 1, 1]], dtype=float)
b = np.array([6.0])

x_plus = np.linalg.pinv(A) @ b
print(f"x+ = {x_plus}  (minimum-norm solution)")
print(f"A @ x+ = {A @ x_plus}  (should be 6)")
print(f"||x+|| = {np.linalg.norm(x_plus):.4f}")

# Compare to another valid solution
x_other = np.array([6, 0, 0], dtype=float)
print(f"\nAnother solution: x = {x_other}, A@x = {A @ x_other}, ||x|| = {np.linalg.norm(x_other):.4f}")
print(f"Pseudoinverse solution has smaller norm: {np.linalg.norm(x_plus) < np.linalg.norm(x_other)}")

## The Polar Decomposition

$A = UH$ where $U$ is orthogonal (rotation) and $H$ is symmetric positive semidefinite (stretch).

Constructed from the SVD: $U = U_{\text{svd}} V^T$, $H = V \Sigma V^T$.

In [ ]:
A = np.array([[0, -2],
              [1,  0]], dtype=float)

U_svd, S, Vt = np.linalg.svd(A)
V = Vt.T

U_polar = U_svd @ Vt         # orthogonal part
H_polar = V @ np.diag(S) @ Vt  # symmetric positive semidefinite part

print("A =")
print(A)
print("\nU (orthogonal):")
print(np.round(U_polar, 6))
print(f"det(U) = {np.linalg.det(U_polar):.4f}  (should be ±1)")
print(f"U^T U = I: {np.allclose(U_polar.T @ U_polar, np.eye(2))}")

print("\nH (symmetric PSD):")
print(np.round(H_polar, 6))
print(f"H symmetric: {np.allclose(H_polar, H_polar.T)}")
print(f"H eigenvalues: {np.round(np.linalg.eigvalsh(H_polar), 6)}  (should be >= 0)")

print(f"\nU @ H (should equal A):\n{np.round(U_polar @ H_polar, 10)}")

In [ ]:
# Visualize the polar decomposition: H stretches, then U rotates
t = np.linspace(0, 2 * np.pi, 300)
circle = np.array([np.cos(t), np.sin(t)])

step1 = H_polar @ circle   # stretch
step2 = U_polar @ step1    # rotate  (= A @ circle)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, data, title in zip(axes,
    [circle, step1, step2],
    ['$x$ (unit circle)', '$Hx$ (stretched)', '$UHx = Ax$ (rotated)']):
    ax.scatter(data[0], data[1], c=t, cmap='hsv', s=4)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=11)
    ax.grid(True, alpha=0.3)
    lim = max(np.max(np.abs(data[0])), np.max(np.abs(data[1]))) * 1.3
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)

plt.suptitle('Polar Decomposition: A = UH', fontsize=13)
plt.tight_layout()
plt.show()

## Summary Table: Everything the SVD Tells You

In [ ]:
A = np.array([[-2, 0],
              [-1, 3]], dtype=float)

U, S, Vt = np.linalg.svd(A)
r = np.sum(S > 1e-10)

print(f"A =\n{A}\n")
print(f"Singular values:   {np.round(S, 6)}")
print(f"Rank:              {r}")
print(f"2-norm:            {S[0]:.6f}")
print(f"Frobenius norm:    {np.sqrt(np.sum(S**2)):.6f}  (cf {np.linalg.norm(A, 'fro'):.6f})")
print(f"Condition number:  {S[0]/S[-1]:.6f}")
print(f"||A^{{-1}}||_2:     {1/S[-1]:.6f}")
print(f"det(A):            {np.linalg.det(A):.6f}  (prod sigma = {np.prod(S):.6f})")
print(f"\nPseudoinverse = Inverse (full rank):")
print(np.round(np.linalg.pinv(A), 6))
print("\nVerify pinv(A) @ A = I:")
print(np.round(np.linalg.pinv(A) @ A, 10))

## Image Compression Revisited

Demonstrating the power of low-rank approximation on a structured synthetic image.

In [ ]:
np.random.seed(1)
n = 300
x = np.linspace(-3, 3, n)
X_g, Y_g = np.meshgrid(x, x)
img = (
    3 * np.exp(-(X_g**2 + Y_g**2) / 2)
    + np.sin(4 * X_g) * np.cos(4 * Y_g)
    + 0.3 * np.random.randn(n, n)
)

U_img, S_img, Vt_img = np.linalg.svd(img)

ranks = [1, 3, 10, 30, 100]
fig, axes = plt.subplots(1, len(ranks) + 1, figsize=(18, 3))

axes[0].imshow(img, cmap='viridis')
axes[0].set_title(f'Original')
axes[0].axis('off')

for ax, k in zip(axes[1:], ranks):
    img_k = (U_img[:, :k] * S_img[:k]) @ Vt_img[:k, :]
    compression = 100 * k * (n + n) / n**2
    ax.imshow(img_k, cmap='viridis')
    ax.set_title(f'Rank {k} ({compression:.0f}%)')
    ax.axis('off')

plt.suptitle('Low-rank approximation via SVD', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Singular value decay
plt.figure(figsize=(7, 3.5))
plt.semilogy(range(1, len(S_img) + 1), S_img, 'o-', markersize=2)
for k in ranks:
    plt.axvline(k, color='gray', linestyle='--', alpha=0.4)
plt.xlabel('Index $k$')
plt.ylabel(r'$\sigma_k$')
plt.title('Singular values of the image')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()